In [41]:
""" import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
"""

' import torch\nprint(torch.cuda.is_available(), torch.cuda.get_device_name(0))\n'

# Create and run a local RAG pipeline

## Goals for RAG

* Retrieval - Find relevant info given a query, e.g. "what are the macronutrients and what do they do?" -> retrieves passages of text realted to the macronutrients from a nutrition textbok.
* Augmented - Take the relevant info and augment our input(prompt) to an LLM with that relevant info
* Generation - Take the first two steps and pass them to an LLM for generative outputs

## Main Goal
improve the outputs of LLMs by:
1. Preventing hallucinations
2. Work with custom data


## Use Cases:
* Textbook Q&A - find releveant passages to the questions you have.

In [42]:
import os
import requests

# Get PDF document path
pdf_path = "sustainability-action-plan.pdf"

# Download PDF
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    #Enter the URL of the PDF
    url = "https://cityofwinterpark.org/docs/departments/natural-resources-sustainability/sustainability-division/sustainability-action-plan/sustainability-action-plan-annual-report-2023.pdf"

    #The local filename to save the downloaded file
    filename=pdf_path

    # Browser-like User-Agent required: the site returns 403 for python-requests' default
    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}


    #Send a GET request to the URL
    response = requests.get(url, headers=headers, timeout=30)

    #Check if the request was successful
    if response.status_code ==200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)

        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. Status code: {response.status_code}")

else:
    print(f"File {pdf_path} exists.")



File sustainability-action-plan.pdf exists.


In [43]:
%pip install tqdm
%pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [44]:
import pymupdf # requires: !pip install PyMuPDF
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text= text.replace("\n"," ").strip()

    return cleaned_text

PRINTED_PAGE_OFFSET = 2
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = pymupdf.open(pdf_path)
    pages_and_texts=[]
    
    for page_number, page in tqdm(enumerate(doc)):
        if page_number < PRINTED_PAGE_OFFSET:
            continue # skip the two pages before the title page
        
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number": page_number - PRINTED_PAGE_OFFSET + 1,
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count": len(text.split(". ")),
                                "page_token_count": len(text) / 4, # 1 token = ~ 4 characters
                                "text": text})

    return pages_and_texts
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

59it [00:00, 80.67it/s]


[{'page_number': 1,
  'page_char_count': 406,
  'page_word_count': 114,
  'page_sentence_count': 1,
  'page_token_count': 101.5,
  'text': 'City of Winter Park  Sustainability Action Plan                  January 24th, 2024                            Presented by:   Gloria Eby, Natural Resources and Sustainability Director     Sara Miller, Sustainability Manager   Mia Brady, Sustainability Specialist   Keep Winter Park Beautiful & Sustainable Advisory Board           Certified WCAG 2.1 Level AA compliant and usable with JAWS and NVDA by SWH'},
 {'page_number': 2,
  'page_char_count': 2873,
  'page_word_count': 155,
  'page_sentence_count': 22,
  'page_token_count': 718.25,
  'text': '2    Contents   Background ........................................................................................................................................... 3  Vision and Purpose ......................................................................................................................

In [45]:
import pandas as pd
pages_and_texts[9]

{'page_number': 10,
 'page_char_count': 1771,
 'page_word_count': 277,
 'page_sentence_count': 4,
 'page_token_count': 442.75,
 'text': '10  o Parks and Recreation summer camp program incorporates Orange County Public Schools (OCPS) Summer Food Program o Summer Feeding - Orange County Public Schools (ocps.net) • Winter Park Library o Master gardening basics:  5 classes, partnership with UF/IFAS Extension orange County o Seed library: 3 grow seasons; 9,000 seeds distributed per year • Rollins College o Food pantry for students – Students can either come to the pantry and take what they need or place an order online, which will be processed at the C-Store and stored in a pre-packed bag o Sam Justice is staff contact - sjustice@rollins.edu o Urban farm collaboration - providing fresh produce to the food pantry on a regular basis • SeniorsFirst o https://www.seniorsfirstinc.org/ o Provides meals-on-wheels to seniors in Winter Park • Winter Park High School o Willies Mart – “Willie’s Mart p

In [46]:

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,text
0,1,406,114,1,101.50,City of Winter Park Sustainability Action Pla...
1,2,2873,155,22,718.25,2 Contents Background ......................
2,3,2448,420,12,612.00,3 Background Vision and Purpose The 2022 Su...
3,4,2954,505,16,738.50,"4 In 2012, the city’s Environmental Review an..."
4,5,2551,411,7,637.75,5 indicators to benchmark city status in effo...


## Table Preprocessing

The goal is to extract tables from the PDF while preserving the relationships between column headers and row values

In [47]:
doc = pymupdf.open(pdf_path)

for page_number, page in enumerate(doc):
    tables = page.find_tables().tables

    if tables:
        print(f"Page {page_number}: {len(tables)} table(s) found")

Page 5: 1 table(s) found
Page 6: 1 table(s) found
Page 12: 1 table(s) found
Page 13: 1 table(s) found
Page 14: 1 table(s) found
Page 15: 1 table(s) found
Page 18: 2 table(s) found
Page 19: 1 table(s) found
Page 20: 1 table(s) found
Page 22: 2 table(s) found
Page 23: 1 table(s) found
Page 25: 2 table(s) found
Page 26: 1 table(s) found
Page 27: 1 table(s) found
Page 28: 1 table(s) found
Page 31: 1 table(s) found
Page 32: 1 table(s) found
Page 33: 1 table(s) found
Page 37: 2 table(s) found
Page 38: 1 table(s) found
Page 39: 1 table(s) found
Page 40: 1 table(s) found
Page 42: 2 table(s) found
Page 43: 1 table(s) found
Page 45: 2 table(s) found
Page 46: 1 table(s) found
Page 47: 1 table(s) found
Page 51: 1 table(s) found
Page 52: 2 table(s) found
Page 55: 2 table(s) found
Page 57: 1 table(s) found


In [48]:
# Automatically extract every table

all_tables = []

for page_number, page in enumerate(doc):
    tables = page.find_tables(strategy="lines_strict").tables

    for table_index, table in enumerate(tables):
        table_df = table.to_pandas()

        all_tables.append({
            "page_number" : page_number,
            "table_index": table_index,
            "data": table_df
        })

print(f"Extracted {len(all_tables)} tables")

Extracted 39 tables


In [49]:
# Fix page offset
for t in all_tables:
    t["printed_page"] = t["page_number"] - PRINTED_PAGE_OFFSET + 1

# check: table 0 was raw page 5, so this should print 4
print(all_tables[0]["page_number"], all_tables[0]["printed_page"])

5 4


# Inspect pages with Tables
## Goals:
1. Turn the tables into dataframe to inspect the actual rows and columns instead of treating the table as one long string

In [50]:
# Inspect the first table
display(all_tables[0]["data"])
# Check which page it came from
print(all_tables[0]["page_number"])

,2012-2013 KWPB&S Board Members,Col1,2014-2015 KWPB&S Board Members,Col3
0,"Mary Dipboye, Chair",James (Bob) Robinson,"Michael Poole, Chair",Mark Roush
1,"Stephen Pategas, V. Chair",Pat Schoknecht,"Stephen Pategas, V. Chair",Bruce Thomas
2,Michele Hipp,Julia Tensfeldt,Michele Hipp,Steven DiClemente
3,Michael Poole,Kent Tse,Raymond Randall,Mary Dipboye
4,Raymond Randall,Laura Walda,Pat Schoknecht,John Tapp
5,John Rife,Carol Kostick,Julia Tensfeldt,Fred Kosiewski
6,Lucy Roberts,Mark Roush,Laura Walda,Cathy Blanton
7,Joseph Robillard,,Carol Shenck (Kostick),


5


In [51]:
df = all_tables[0]["data"].copy()

# Combine each pair of columns vertically and remove empty cells
board_2012 = pd.concat([df.iloc[:, 0], df.iloc[:, 1]]).astype(str).str.strip()
board_2012 = board_2012[board_2012.ne("") & board_2012.ne("nan")].reset_index(drop=True)

board_2014 = pd.concat([df.iloc[:, 2], df.iloc[:, 3]]).astype(str).str.strip()
board_2014 = board_2014[board_2014.ne("") & board_2014.ne("nan")].reset_index(drop=True)

# Create the cleaned table
cleaned_df = pd.DataFrame({
    "2012-2013 KWPB&S Board Members": board_2012,
    "2014-2015 KWPB&S Board Members": board_2014
})

# Save it back into all_tables
all_tables[0]["data"] = cleaned_df

display(all_tables[0]["data"])

,2012-2013 KWPB&S Board Members,2014-2015 KWPB&S Board Members
0,"Mary Dipboye, Chair","Michael Poole, Chair"
1,"Stephen Pategas, V. Chair","Stephen Pategas, V. Chair"
2,Michele Hipp,Michele Hipp
3,Michael Poole,Raymond Randall
4,Raymond Randall,Pat Schoknecht
5,John Rife,Julia Tensfeldt
6,Lucy Roberts,Laura Walda
7,Joseph Robillard,Carol Shenck (Kostick)
8,James (Bob) Robinson,Mark Roush
9,Pat Schoknecht,Bruce Thomas


In [52]:
display(all_tables[13]["data"])
print(all_tables[13]["printed_page"])

,Col0,Indicator Description,2012\nBaseline,2021\nStatus,2022\nStatus,2025\nTarget,2035\nTarget
0,W-1,Water usage in residential buildings\n[gallons...,"127,777","130,849","127,606",TBD,TBD
1,W-2,Water usage in commercial buildings\n[gallons/...,"294,098","303,788","315,099",TBD,TBD
2,W-3,Reclaimed water usage [million\ngallons/year]2,144.5,91.61,87.97,10%\nincrease,30%\nincrease
3,W-4,Residential Rebates provided\nannually,-,34,19,75,150


24


In [53]:
display(all_tables[32]["data"])
print(all_tables[32]["page_number"])

,Climate Resiliency,Conduct Renewable Energy Feasibility Study
0,Climate Resiliency,Implement an Artificial Turf Ordinance which w...
1,Climate Resiliency,Adopt Backyard Chicken Program (exp. September...
2,Water,Create water conservation education to residen...
3,Water,Increase public awareness of Florida-friendly ...
4,Community Engagement and Green\nEconomy,Re-participate in America In Bloom’s annual na...
5,Local Government Operations,Solicit proposals for energy conservation audi...
6,Local Government Operations,Conduct Renewable Energy Feasibility Study – 5...
7,Natural Resources,Provide education on pollution prevention of n...
8,Natural Resources,Establish Lakes Newsletter to inform citizens ...
9,Waste Management,Provide composting education and backyard comp...


47


In [54]:
display(all_tables[38]["data"])
print(all_tables[38]["page_number"])

,Year,Annualized\nCost\n($/MWh),Annualized\nCost\n($/MWh)\nescalated at\n3% after\n2042,LCOE\n($/MWh)\nwith a 3%\nescalation,Maximum\nAcceptable\nAnnualized\nPower Costs
0,2023,$72,,$70,
1,2024,$72,,$73,
2,2025,$72,,$75,
3,2026,$74,,$77,
4,2027,$75,,$79,
5,2028,$77,,$82,
6,2029,$78,,$84,
7,2030,$77,,$87,
8,2031,$76,,$89,
9,2032,$80,,$92,


57


In [55]:
import random
random.sample(pages_and_texts, k=3)

[{'page_number': 49,
  'page_char_count': 3279,
  'page_word_count': 529,
  'page_sentence_count': 20,
  'page_token_count': 819.75,
  'text': '49  a framework for healthy, highly efficient, and cost-saving green buildings. LEED certification  is a globally recognized symbol of sustainability achievement and leadership.   Open Space is any open piece of land that is undeveloped and is accessible to the public.   In your community, there could be many creative opportunities for open space  preservation that could help connect the community and revitalize its economy and social  connectivity.  Some opportunities for community open space can include: playgrounds,  public plazas, or vacant lots as well as Green space (land that is partly or completely  covered with grass, trees, shrubs, or other vegetation) such as parks, community gardens,  and cemeteries.  Pay As You Throw is a system in which residents pay for municipal solid waste (MSW)  services per unit of waste discarded rather than

In [56]:
df.describe().round(2)

,2012-2013 KWPB&S Board Members,Col1,2014-2015 KWPB&S Board Members,Col3
count,8,8,8,8
unique,8,8,8,8
top,"Mary Dipboye, Chair",James (Bob) Robinson,"Michael Poole, Chair",Mark Roush
freq,1,1,1,1


# Further text processing (splitting pages into sentences)

In [57]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [58]:
from spacy.lang.en import English

nlp = English()
nlp.add_pipe("sentencizer")

In [59]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    #Make sure all sentences are strings (default type is spaCy datatype)
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    #Count the sentences
    item["page_sentence_count_spacy"] = len(item["sentences"])

100%|██████████| 57/57 [00:00<00:00, 125.47it/s]


In [60]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,page_sentence_count_spacy
count,57.0,57.00,57.00,57.00,57.00,57.00
mean,29.0,2050.11,353.60,9.28,512.53,8.79
std,16.6,701.39,113.62,7.93,175.35,7.58
min,1.0,406.00,114.00,1.00,101.50,1.00
25%,15.0,1559.00,279.00,2.00,389.75,2.00
50%,29.0,1958.00,366.00,7.00,489.50,7.00
75%,43.0,2420.00,420.00,15.00,605.00,14.00
max,57.0,3876.00,639.00,33.00,969.00,32.00


## Remove the Tables from the Prose track

In [ ]:
# find where each table sits on the page
table_bboxes_by_page = {}

for page_number, page in enumerate(doc):
    tables = page.find_tables(strategy="lines_strict").tables
    if tables:
        table_bboxes_by_page[page_number] = [t.bbox for t in tables]

In [62]:
print(sum(len(v) for v in table_bboxes_by_page.values()), len(all_tables))

39 39


In [66]:
# rebuild each page's text, leave out anything inside the table
# check for each block to see how much is in the table, drop iff half of the block is inside
def mostly_inside(block_rect, bboxes, threshold=0.5):
    block = pymupdf.Rect(block_rect)
    area = block.get_area()
    if area == 0:
        return False
    return any((block & pymupdf.Rect(bb)).get_area() / area > threshold for bb in bboxes)

pages_and_texts_notables = []

for page_number, page in tqdm(enumerate(doc)):
    if page_number < PRINTED_PAGE_OFFSET:
        continue

    bboxes = table_bboxes_by_page.get(page_number, [])
    blocks = page.get_text("blocks", sort=True)
    kept = [b[4] for b in blocks if b[6] == 0 and not mostly_inside(b[:4], bboxes)]

    text = text_formatter(" ".join(kept))
    pages_and_texts_notables.append({
        "page_number": page_number - PRINTED_PAGE_OFFSET + 1,
        "page_char_count": len(text),
        "page_word_count": len(text.split(" ")),
        "page_sentence_count": len(text.split(". ")),
        "page_token_count": len(text) / 4, # 1 token = ~ 4 characters
        "text": text,
    })

59it [00:00, 59.54it/s]


In [ ]:
old = {p["page_number"]: p["page_char_count"] for p in pages_and_texts}
for p in pages_and_texts_notables:
    pg = p["page_number"]
    drop = 1 - p["page_char_count"] / old[pg] if old[pg] else 0
    if drop > 0.4:
        print(f"Page {pg}: lost {drop:.0%} of its text")

Page 12: lost 94% of its text
Page 13: lost 95% of its text
Page 14: lost 95% of its text
Page 17: lost 94% of its text
Page 18: lost 95% of its text
Page 19: lost 91% of its text
Page 21: lost 72% of its text
Page 22: lost 95% of its text
Page 24: lost 67% of its text
Page 25: lost 96% of its text
Page 27: lost 95% of its text
Page 31: lost 95% of its text
Page 32: lost 96% of its text
Page 36: lost 89% of its text
Page 37: lost 95% of its text
Page 38: lost 95% of its text
Page 39: lost 78% of its text
Page 41: lost 53% of its text
Page 42: lost 97% of its text
Page 44: lost 53% of its text
Page 45: lost 96% of its text
Page 46: lost 94% of its text
Page 56: lost 44% of its text


In [68]:
df = pd.DataFrame(pages_and_texts_notables)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count
count,57.0,57.00,57.00,57.00,57.00
mean,29.0,1385.98,227.86,8.96,346.50
std,16.6,1138.51,185.08,8.13,284.63
min,1.0,78.00,16.00,1.00,19.50
25%,15.0,102.00,23.00,1.00,25.50
50%,29.0,1438.00,235.00,7.00,359.50
75%,43.0,2259.00,390.00,15.00,564.75
max,57.0,3882.00,645.00,33.00,970.50


### Chunking sentences together
1. Makes texts easier to filter
2. Allows text can fit into embedding model context window
3. Context passed to an LLM can be more specific and focused. 

In [71]:
for item in tqdm(pages_and_texts_notables):
    item["sentences"] = list(nlp(item["text"]).sents)

    #Make sure all sentences are strings (default type is spaCy datatype)
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    #Count the sentences
    item["page_sentence_count_spacy"] = len(item["sentences"])

100%|██████████| 57/57 [00:00<00:00, 179.81it/s]


In [ ]:

# Create a function to split lists of texts into chunk size
def split_list_overlap(input_list, slice_size=5, overlap=1):
    step = slice_size - overlap
    return [input_list[i:i+slice_size] for i in range(0, len(input_list), step)]

test_list = list(range(25))

In [ ]:
# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts_notables):
    item["sentence_chunks"] = split_list_overlap(item["sentences"], slice_size=5, overlap=1)
    item["num_chunks"] = len(item["sentence_chunks"]) 


100%|██████████| 57/57 [00:00<00:00, 176961.75it/s]


In [28]:
random.sample(pages_and_texts,k=1)

[{'page_number': 21,
  'page_char_count': 1958,
  'page_word_count': 386,
  'page_sentence_count': 7,
  'page_token_count': 489.5,
  'text': '21  OBJECTIVES  1. Increase energy efficiency of residential and commercial buildings 2. Increase energy conservation in residential and commercial sectors 3. Increase solar energy in residential and commercial sectors 4. Increase educational awareness for energy efficiency and conservation best practices 5. Maintain utility rates that are competitively low for consumers INDICATORS  Indicator Description  Baseline  2021  Status  2022  Status  2025  Target  2035  Target  2042  Target  2050  Target  E-1 Energy usage in residential  buildings   [kWh/customer/year]1 –   Baseline Year: 2012   15,263   14,739   14,791  13,718  12,064  13,560  12,710  E-2 Energy usage in commercial  buildings [kWh/customer/year] 1 –  Baseline Year: 2012   -   86,704   91,527  81,004  73,312  73,858    70,838  E-3 Residential  building  audits  performed annually – Basel

In [29]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,page_sentence_count_spacy,num_chunks
count,57.0,57.00,57.00,57.00,57.00,57.00,57.00
mean,29.0,2050.11,353.60,9.28,512.53,8.79,1.44
std,16.6,701.39,113.62,7.93,175.35,7.58,0.71
min,1.0,406.00,114.00,1.00,101.50,1.00,1.00
25%,15.0,1559.00,279.00,2.00,389.75,2.00,1.00
50%,29.0,1958.00,366.00,7.00,489.50,7.00,1.00
75%,43.0,2420.00,420.00,15.00,605.00,14.00,2.00
max,57.0,3876.00,639.00,33.00,969.00,32.00,4.00


### Things to check later: remove the tables' text from the prose track

### Splitting each chunk into its own item



In [ ]:
import re

# Split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts_notables):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # Join the sentences together into a pragraph-like strucutre(join list of sentences into one paragraph)
        joined_sentence_chunk = "".join(sentence_chunk).replace(" "," ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)


        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Get some stats on our chunks
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 chars

        pages_and_chunks.append(chunk_dict)


100%|██████████| 57/57 [00:00<00:00, 10726.64it/s]


In [34]:
random.sample(pages_and_texts, k=1)

[{'page_number': 2,
  'page_char_count': 2873,
  'page_word_count': 155,
  'page_sentence_count': 22,
  'page_token_count': 718.25,
  'text': '2    Contents   Background ........................................................................................................................................... 3  Vision and Purpose ......................................................................................................................... 3  History ............................................................................................................................................... 3  About the 2022 SAP Update ........................................................................................................ 4  Overview ......................................................................................................................................... 4  2021 SAP Update Community Engagement Process ...........................................................